In [2]:
!pip install -q langchain langchain-groq gspread

In [3]:
import os
from google.colab import userdata, auth

# Load Groq API key from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

print("Groq connected successfully!")

Groq connected successfully!


In [4]:
from langchain.tools import tool

BOOKS = {
    "python crash course": {
        "author": "Eric Matthes",
        "category": "Programming",
        "available": True,
        "copies": 3
    },

    "let us c": {
        "author": "Yashavant Kanetkar",
        "category": "Programming",
        "available": True,
        "copies": 2
    },

    "clean code": {
        "author": "Robert C. Martin",
        "category": "Software Engineering",
        "available": False,
        "copies": 0
    },

    "data structures and algorithms": {
        "author": "Narasimha Karumanchi",
        "category": "Computer Science",
        "available": True,
        "copies": 4
    },

    "operating system concepts": {
        "author": "Abraham Silberschatz",
        "category": "Computer Science",
        "available": True,
        "copies": 2
    }
}


# TOOL 1: Get book details

@tool
def get_book_details(title: str) -> str:
    """
    Get the author, category and other details of a book
    from the college library catalogue.

    Args:
        title: The title of the book.
    """

    book = BOOKS.get(title.lower().strip())

    if book is None:
        return f"Sorry, {title} was not found in the library catalogue."

    return (
        f"Book: {title}\n"
        f"Author: {book['author']}\n"
        f"Category: {book['category']}\n"
        f"Available: {book['available']}\n"
        f"Copies: {book['copies']}"
    )


# TOOL 2: Check book availability

@tool
def check_availability(title: str) -> str:
    """
    Check whether a particular book is currently available
    in the college library.

    Args:
        title: The title of the book.
    """

    book = BOOKS.get(title.lower().strip())

    if book is None:
        return f"{title} is not present in the catalogue."

    if book["available"]:
        return (
            f"Yes, {title} is available. "
            f"Copies available: {book['copies']}."
        )

    return f"Sorry, {title} is currently unavailable."


print("Both library tools created successfully!")

Both library tools created successfully!


In [6]:
from pydantic import BaseModel, Field
from typing import Literal, Optional


class LibraryRequest(BaseModel):
    """Structured record of a student's library request."""

    student_name: Optional[str] = Field(
        default=None,
        description="Name of the student"
    )

    book_title: Optional[str] = Field(
        default=None,
        description="Title of the requested book"
    )

    request_type: Literal[
        "availability",
        "borrow_request",
        "general_enquiry"
    ] = Field(
        default="general_enquiry",
        description="Type of library request"
    )

    preferred_date: Optional[str] = Field(
        default=None,
        description="Preferred date for borrowing or visiting"
    )

    notes: Optional[str] = Field(
        default=None,
        description="Any additional information"
    )


print("Pydantic LibraryRequest model created!")

Pydantic LibraryRequest model created!


In [11]:
extractor = llm.with_structured_output(
    LibraryRequest,
    method="json_mode"
)

messy_messages = [
    "hi im ravi can i borrow python crash course tomorrow",

    "this is priya, is clean code available?",

    "hello, i want to know about let us c book",

    "im arjun can i request data structures and algorithms for monday",

    "hey, my name is kavya, i need operating system concepts"
]

for i, message in enumerate(messy_messages, start=1):

    print("\nTEST", i)
    print("Input:", message)

    # Using a clean format without escaped nested quotes to prevent IncompleteInputError
    prompt = f"""You are an extraction assistant designed to output a structured JSON object matching this schema:
- student_name: Name of the student (or null if not mentioned)
- book_title: Title of the requested book (or null if not mentioned)
- request_type: Must be exactly one of 'availability', 'borrow_request', or 'general_enquiry'. Do not use any other value.
- preferred_date: Preferred date for borrowing or visiting (or null if not mentioned)
- notes: Any additional information (or null)

Extract from this user message:
{message}"""

    result = extractor.invoke(prompt)

    print("Structured output:")
    print(result.model_dump())


TEST 1
Input: hi im ravi can i borrow python crash course tomorrow
Structured output:
{'student_name': 'Ravi', 'book_title': 'Python Crash Course', 'request_type': 'borrow_request', 'preferred_date': 'tomorrow', 'notes': None}

TEST 2
Input: this is priya, is clean code available?
Structured output:
{'student_name': 'Priya', 'book_title': 'Clean Code', 'request_type': 'availability', 'preferred_date': None, 'notes': None}

TEST 3
Input: hello, i want to know about let us c book
Structured output:
{'student_name': None, 'book_title': 'Let Us C', 'request_type': 'general_enquiry', 'preferred_date': None, 'notes': 'i want to know about let us c book'}

TEST 4
Input: im arjun can i request data structures and algorithms for monday
Structured output:
{'student_name': 'Arjun', 'book_title': 'Data Structures and Algorithms', 'request_type': 'borrow_request', 'preferred_date': 'Monday', 'notes': None}

TEST 5
Input: hey, my name is kavya, i need operating system concepts
Structured output:
{'

In [13]:
import gspread
from google.colab import auth
from google.auth import default

auth.authenticate_user()

creds, _ = default()
gc = gspread.authorize(creds)

SHEET_NAME = "ABC College Library Requests"

# Open an existing sheet or create a new one
try:
    sh = gc.open(SHEET_NAME)
    print("Existing Google Sheet opened.")

except gspread.SpreadsheetNotFound:
    sh = gc.create(SHEET_NAME)
    print("New Google Sheet created.")

ws = sh.sheet1

headers = [
    "Student Name",
    "Book Title",
    "Request Type",
    "Preferred Date",
    "Notes",
    "Status"
]

# Add headers only if the sheet is empty
if not ws.get_all_values():
    ws.append_row(headers)

print("Google Sheet is ready!")
print("Sheet link:", sh.url)

New Google Sheet created.
Google Sheet is ready!
Sheet link: https://docs.google.com/spreadsheets/d/1CkEFpUSsrW89jnLssR4h5iyldmDAO8mCCVSmg3ru_HU


In [14]:
@tool
def log_library_request(
    student_name: str,
    book_title: str,
    request_type: str,
    preferred_date: str = "",
    notes: str = ""
) -> str:
    """
    Save a student's library request to the Google Sheet.
    Use this only when the student explicitly asks to
    submit or record a request.

    Args:
        student_name: Student's name.
        book_title: Requested book title.
        request_type: availability, borrow_request or general_enquiry.
        preferred_date: Preferred date, if provided.
        notes: Additional information.
    """

    try:

        request = LibraryRequest(
            student_name=student_name,
            book_title=book_title,
            request_type=request_type,
            preferred_date=preferred_date or None,
            notes=notes or None
        )

        if not request.student_name or not request.student_name.strip():
            return "Student name is required."

        if not request.book_title or not request.book_title.strip():
            return "Book title is required."

        ws.append_row([
            request.student_name,
            request.book_title,
            request.request_type,
            request.preferred_date or "",
            request.notes or "",
            "New"
        ])

        return (
            f"Request saved successfully for "
            f"{request.student_name}."
        )

    except Exception as e:
        return f"Could not save request: {str(e)}"


print("Google Sheets logging tool created!")

Google Sheets logging tool created!


In [15]:
from langchain.agents import create_agent

SYSTEM_PROMPT = """
You are a friendly College Library Assistant.

Your responsibilities:

1. Answer students' questions about library books.
2. Use get_book_details to find book authors and categories.
3. Use check_availability to check whether a book is available.
4. Never invent book information.
5. If a book is not in the catalogue, explain that clearly.
6. Library working hours are Monday to Saturday,
   9:00 AM to 5:00 PM.
7. Sunday is a holiday.
8. Use log_library_request only when the student
   explicitly asks you to submit or record a request.
9. Before logging, collect the student's name and book title.
10. Never claim that a book has actually been issued.
    Logging a request is not the same as issuing a book.

Keep answers simple, polite and helpful.
"""

agent = create_agent(
    model=llm,
    tools=[
        get_book_details,
        check_availability,
        log_library_request
    ],
    system_prompt=SYSTEM_PROMPT
)

print("College Library AI Agent created successfully!")

College Library AI Agent created successfully!


In [16]:
history = []

def chat(question):

    global history

    try:

        result = agent.invoke({
            "messages": history + [
                {
                    "role": "user",
                    "content": question
                }
            ]
        })

        history = result["messages"]

        return history[-1].content

    except Exception as e:

        print("Error:", e)
        return "Sorry, something went wrong. Please try again."


print(chat("Is Python Crash Course available?"))

print(chat("Who is the author of Clean Code?"))

print(chat("What are the library timings on Sunday?"))

print(chat(
    "My name is Ravi. Please submit a borrow request "
    "for Python Crash Course for tomorrow."
))

print(chat(
    "I am Priya. Please record my request to check "
    "the availability of Let Us C."
))

Yes! “Python Crash Course” by Eric Matthes is in the catalog and currently available. There are 3 copies in stock. If you’d like to reserve one, just let me know!
The author of **“Clean Code”** is **Robert C. Martin**.
The library is closed on Sundays—our hours are Monday to Saturday, 9:00 AM to 5:00 PM.
Your borrow request for **“Python Crash Course”** has been recorded for tomorrow. If you have any other questions or need further assistance, just let me know!
Your request to check the availability of **“Let Us C”** has been recorded. If you need any more help, feel free to ask!


In [17]:
@tool
def tool1(title: str) -> str:
    """does stuff"""

    book = BOOKS.get(title.lower().strip())

    if book is None:
        return "Book not found."

    return (
        f"Book: {title}, "
        f"Author: {book['author']}, "
        f"Category: {book['category']}"
    )


bad_agent = create_agent(
    model=llm,
    tools=[
        tool1,
        check_availability
    ],
    system_prompt="""
    You are a library assistant.
    Use the available tools to answer book questions.
    """
)

bad_result = bad_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Who is the author of Clean Code?"
        }
    ]
})

for message in bad_result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Who is the author of Clean Code?
================================== Ai Message ==================================

The book **“Clean Code”** was written by **Robert C. Martin** (often referred to as “Uncle Bob”).


In [18]:
# Ask one question
trace_result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Is Clean Code available in the library?"
        }
    ]
})

# Display complete trace
for message in trace_result["messages"]:
    message.pretty_print()


# Count model calls and tool calls
from langchain_core.messages import AIMessage, ToolMessage

model_calls = 0
tool_calls = 0
tool_results = 0

for message in trace_result["messages"]:

    if isinstance(message, AIMessage):
        model_calls += 1
        tool_calls += len(message.tool_calls)

    elif isinstance(message, ToolMessage):
        tool_results += 1


print("\n----- EXECUTION SUMMARY -----")
print("Model calls:", model_calls)
print("Tool calls:", tool_calls)
print("Tool results:", tool_results)

================================ Human Message =================================

Is Clean Code available in the library?
================================== Ai Message ==================================
Tool Calls:
  check_availability (fc_9353277d-5312-4d2c-b04a-3a9a43c41474)
 Call ID: fc_9353277d-5312-4d2c-b04a-3a9a43c41474
  Args:
    title: Clean Code
================================= Tool Message =================================
Name: check_availability

Sorry, Clean Code is currently unavailable.
================================== Ai Message ==================================
Tool Calls:
  get_book_details (fc_413d7213-cba1-4757-9f57-829061dd49a9)
 Call ID: fc_413d7213-cba1-4757-9f57-829061dd49a9
  Args:
    title: Clean Code
================================= Tool Message =================================
Name: get_book_details

Book: Clean Code
Author: Robert C. Martin
Category: Software Engineering
Available: False
Copies: 0
================================== Ai Message ======

In [19]:
library_facts = """
ABC COLLEGE LIBRARY - INFORMATION

1. The library is called ABC College Library.
2. The library is open from Monday to Saturday.
3. The library opens at 9:00 AM.
4. The library closes at 5:00 PM.
5. Sunday is a holiday.
6. Python Crash Course is available in the sample catalogue.
7. Python Crash Course is written by Eric Matthes.
8. Python Crash Course belongs to the Programming category.
9. Let Us C is written by Yashavant Kanetkar.
10. Let Us C belongs to the Programming category.
11. Clean Code is written by Robert C. Martin.
12. Clean Code belongs to Software Engineering.
13. Clean Code is currently unavailable in the sample catalogue.
14. Data Structures and Algorithms is written by Narasimha Karumanchi.
15. Operating System Concepts is written by Abraham Silberschatz.
"""

with open("library_facts.txt", "w") as file:
    file.write(library_facts)

print("Library facts file created successfully!")

from google.colab import files
files.download("library_facts.txt")

Library facts file created successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>